# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane:** Refresh / Content Opportunity Scoring  
**Dataset:** `data/raw/content_refresh_anonymized.csv` (30,000 rows × 44 columns)  
**Label:** `is_declining` = 1 when `trend_direction == "down"` (base rate ≈ 54.2%)

In [1]:
# ── Imports and data load ──────────────────────────────────────
import pandas as pd
import numpy as np
import os
from IPython.display import display, Markdown

# Load the starter dataset (one row per content item, trailing-90-day metrics)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Derive the binary label the same way the prep script does
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
print(f"Base rate of declining: {df['is_declining'].mean():.3f}")

Loaded 30,000 rows × 45 columns
Base rate of declining: 0.542


---
## Signal check — two hypotheses before building the rule

Before writing a rule, we test two signals that should matter for content refresh.
Each signal gets a bucket table with `n` per bucket and a verdict.

### Signal 1: Staleness (`days_since_last_update`)

**Hypothesis:** Pages that have not been updated in a long time are more likely to
be declining. This connects to the FlyRank concept of *staleness* — content that
hasn't been refreshed becomes outdated and loses search visibility.

Leakage note: `days_since_last_update` is a **historical content property** — it
tells us when the page was last edited, NOT how it will perform in the future.
This is safe to use as a feature.

In [2]:
# ── Signal 1: Staleness vs decline rate ────────────────────────
bins_stale = [0, 30, 90, 180, 400]
labels_stale = ["0–30 d", "31–90 d", "91–180 d", "181+ d"]
df["stale_bucket"] = pd.cut(
    df["days_since_last_update"], bins=bins_stale,
    labels=labels_stale, right=True
)

signal1 = (
    df.groupby("stale_bucket", observed=True)
    .agg(n=("is_declining", "count"), decline_rate=("is_declining", "mean"))
    .reset_index()
)
signal1["decline_rate"] = signal1["decline_rate"].round(3)

print("Signal 1 — Staleness vs decline rate")
print(f"(base rate = {df['is_declining'].mean():.3f})")
print()
print(signal1.to_string(index=False))

Signal 1 — Staleness vs decline rate
(base rate = 0.542)

stale_bucket     n  decline_rate
      0–30 d 20480         0.511
     31–90 d   175         0.589
    91–180 d  9171         0.611
      181+ d   174         0.471


In [3]:
display(Markdown("""
**Signal 1 verdict: CONFIRMED (directional)**

The decline rate rises from **0.511** in recently-updated pages (0–30 d) to
**0.611** for pages not updated in 91–180 days — a ~10 percentage-point lift
above the base rate of 0.542. The 181+ d bucket is small (n=174) and dips,
so the signal is directional rather than monotonic. Overall, staleness
is a confirmed, if moderate, predictor of decline in this dataset.
"""))


**Signal 1 verdict: CONFIRMED (directional)**

The decline rate rises from **0.511** in recently-updated pages (0–30 d) to
**0.611** for pages not updated in 91–180 days — a ~10 percentage-point lift
above the base rate of 0.542. The 181+ d bucket is small (n=174) and dips,
so the signal is directional rather than monotonic. Overall, staleness
is a confirmed, if moderate, predictor of decline in this dataset.


### Signal 2: CTR vs Position (the "striking distance" concept)

**Hypothesis:** Pages sitting just outside top-10 (the "striking distance"
positions 11–20) have low CTR and are likely declining — they are visible
enough to generate impressions but not ranked high enough to earn clicks.
This connects to the FlyRank concept of *CTR vs position* — a page
with many impressions but poor click-through is a prime refresh candidate.

Leakage note: `avg_position` and `ctr` are trailing-90-day metrics describing
the *observation window* — they describe where the page sat historically.
They are NOT derived from the label (`trend_direction` / `trend_pct`).

In [4]:
# ── Signal 2: CTR vs Position ──────────────────────────────────
# Exclude avg_position == 0 ("no data", per data dictionary)
df_pos = df[df["avg_position"] > 0].copy()

pos_bins = [0, 3, 10, 20, 50, 250]
pos_labels = ["top 3", "page 1 (4–10)", "striking (11–20)",
              "page 3–5 (21–50)", "deep (50+)"]
df_pos["pos_bucket"] = pd.cut(
    df_pos["avg_position"], bins=pos_bins,
    labels=pos_labels, right=True
)

signal2 = (
    df_pos.groupby("pos_bucket", observed=True)
    .agg(
        n=("ctr", "count"),
        mean_ctr=("ctr", "mean"),
        decline_rate=("is_declining", "mean"),
    )
    .reset_index()
)
signal2["mean_ctr"] = signal2["mean_ctr"].round(3)
signal2["decline_rate"] = signal2["decline_rate"].round(3)

print("Signal 2 — Position bucket vs CTR and decline rate")
print(f"(base rate = {df['is_declining'].mean():.3f}, "
      f"rows with position data = {len(df_pos):,})")
print()
print(signal2.to_string(index=False))

Signal 2 — Position bucket vs CTR and decline rate
(base rate = 0.542, rows with position data = 28,795)

      pos_bucket     n  mean_ctr  decline_rate
           top 3  1141     2.714         0.498
   page 1 (4–10) 11842     0.651         0.569
striking (11–20)  7273     0.323         0.610
page 3–5 (21–50)  7225     0.222         0.562
      deep (50+)  1314     0.151         0.343


In [5]:
display(Markdown("""
**Signal 2 verdict: CONFIRMED (directional)**

CTR drops sharply as position worsens — from **2.71%** in top-3 to **0.15%**
in deep (50+). The decline rate peaks at the *striking-distance* bucket
(11–20) at **0.610**, well above the base rate. Pages ranked just outside
page 1 are the most likely to be declining: they generate impressions but
few clicks. The deep bucket has a *lower* decline rate (0.343), which
makes sense — pages already ranked poorly have less room to fall.

The signal is not perfectly monotonic (top-3 is below base rate, deep is
below too), but the actionable middle — striking-distance pages — is
clearly the worst off. Verdict: confirmed.
"""))


**Signal 2 verdict: CONFIRMED (directional)**

CTR drops sharply as position worsens — from **2.71%** in top-3 to **0.15%**
in deep (50+). The decline rate peaks at the *striking-distance* bucket
(11–20) at **0.610**, well above the base rate. Pages ranked just outside
page 1 are the most likely to be declining: they generate impressions but
few clicks. The deep bucket has a *lower* decline rate (0.343), which
makes sense — pages already ranked poorly have less room to fall.

The signal is not perfectly monotonic (top-3 is below base rate, deep is
below too), but the actionable middle — striking-distance pages — is
clearly the worst off. Verdict: confirmed.


---
## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule in plain words:**
> "A page is worth refreshing if it is **stale** (not updated in ≥ 91 days)
> AND it is **visible** (≥ 100 impressions in 90 days) AND it sits in
> **striking distance** (avg position 4–20, i.e. page 1 or striking-distance).
> The score is the product: `stale × visible × impressions_90d`.
> Higher impressions push it up because there's more traffic to recover."

**Reason codes:**
- `stale_and_visible` — the page is stale, has traffic, and sits in striking distance

**Action label:** `REFRESH` (for every page the rule flags with score > 0)

In [6]:
# ── 1. Build the rule (transparent, no fitted weights) ─────────
# Staleness flag: not updated in 91+ days
df["is_stale"] = (df["days_since_last_update"] >= 91).astype(int)

# Visibility flag: at least 100 impressions in the 90-day window
df["is_visible"] = (df["impressions_90d"] >= 100).astype(int)

# Striking-distance flag: avg_position between 4 and 20 (page 1 or striking)
# Exclude avg_position == 0 (no data)
df["is_striking"] = (
    (df["avg_position"] >= 4) & (df["avg_position"] <= 20)
).astype(int)

# Score: product of binary flags × impressions (higher impressions = more upside)
df["score"] = df["is_stale"] * df["is_visible"] * df["is_striking"] * df["impressions_90d"]

# Reason code (exactly one per row)
df["reason_code"] = np.where(df["score"] > 0, "stale_and_visible", "not_flagged")

# Action label
df["action"] = np.where(df["score"] > 0, "REFRESH", "HOLD")

flagged = df["score"] > 0
print(f"Rule flags {flagged.sum():,} of {len(df):,} pages ({flagged.mean():.1%})")
print(f"Flagged pages — decline rate: {df.loc[flagged, 'is_declining'].mean():.3f}")
print(f"Un-flagged pages — decline rate: {df.loc[~flagged, 'is_declining'].mean():.3f}")

Rule flags 4,659 of 30,000 pages (15.5%)
Flagged pages — decline rate: 0.638
Un-flagged pages — decline rate: 0.524


---
## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# ── 2. Rank and write the CSV ──────────────────────────────────
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)
df_ranked = df.sort_values("rank")

# Select output columns
out_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action",
    "days_since_last_update", "impressions_90d", "avg_position", "ctr",
    "is_declining",
]
df_out = df_ranked[out_cols].copy()

# Write to CSV
os.makedirs("../../work/outputs", exist_ok=True)
csv_path = "../../work/outputs/baseline_action_score.csv"
df_out.to_csv(csv_path, index=False)
print(f"Wrote {len(df_out):,} rows to {csv_path}")
print()
print("Top 5 preview:")
print(df_out.head().to_string(index=False))

Wrote 30,000 rows to ../../work/outputs/baseline_action_score.csv

Top 5 preview:
 rank           content_id         client_id  score       reason_code  action  days_since_last_update  impressions_90d  avg_position  ctr  is_declining
    1 content_5fe46e04994d client_4e07408562 517715 stale_and_visible REFRESH                     104           517715           4.2 0.14             1
    2 content_2c2606c5d176 client_19581e27de 347399 stale_and_visible REFRESH                     104           347399           4.2 0.53             1
    3 content_cb112fce36be client_19581e27de 309910 stale_and_visible REFRESH                     104           309910           5.6 0.16             1
    4 content_36ff89c8214e client_19581e27de 295097 stale_and_visible REFRESH                     104           295097           7.3 0.05             0
    5 content_c21024970297 client_19581e27de 211366 stale_and_visible REFRESH                     104           211366           5.1 0.41             0


---
### Baseline evaluation: precision@K

Per the `building-baselines` skill: the honest metric for a "which ones first?"
problem is **precision@K** — of the top K the rule flags, how many were actually
declining? Always print the base rate next to it.

In [8]:
# ── Precision@K evaluation ─────────────────────────────────────
def precision_at_k(scores, labels, k):
    """Of the top-K scored items, what fraction have label == 1?"""
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining"].mean()

for k in [10, 20, 50, 100]:
    p_at_k = precision_at_k(df["score"].values, df["is_declining"].values, k)
    print(f"precision@{k:>3d} = {p_at_k:.3f}   (base rate = {base_rate:.3f},  "
          f"lift = {p_at_k - base_rate:+.3f})")

precision@ 10 = 0.500   (base rate = 0.542,  lift = -0.042)
precision@ 20 = 0.450   (base rate = 0.542,  lift = -0.092)
precision@ 50 = 0.380   (base rate = 0.542,  lift = -0.162)
precision@100 = 0.360   (base rate = 0.542,  lift = -0.182)


---
## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# ── 3. Top-10 hand review ──────────────────────────────────────
top10 = df_ranked.head(10)[
    ["rank", "content_id", "score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "avg_position",
     "ctr", "is_declining"]
].copy()

print("Top-10 ranked pages:")
print()
print(top10.to_string(index=False))

Top-10 ranked pages:

 rank           content_id  score       reason_code  action  days_since_last_update  impressions_90d  avg_position  ctr  is_declining
    1 content_5fe46e04994d 517715 stale_and_visible REFRESH                     104           517715           4.2 0.14             1
    2 content_2c2606c5d176 347399 stale_and_visible REFRESH                     104           347399           4.2 0.53             1
    3 content_cb112fce36be 309910 stale_and_visible REFRESH                     104           309910           5.6 0.16             1
    4 content_36ff89c8214e 295097 stale_and_visible REFRESH                     104           295097           7.3 0.05             0
    5 content_c21024970297 211366 stale_and_visible REFRESH                     104           211366           5.1 0.41             0
    6 content_c8e9d6ab9013 208678 stale_and_visible REFRESH                     104           208678           9.7 0.00             1
    7 content_d17681677e69 201584 stale_

In [10]:
# ── 3b. Detailed commentary for each top-10 row ───────────────
for _, row in top10.iterrows():
    actual = "DECLINING" if row["is_declining"] == 1 else "NOT declining"
    days = int(row["days_since_last_update"])
    impr = int(row["impressions_90d"])
    pos = row["avg_position"]
    ctr_val = row["ctr"]

    print(f"── Rank {int(row['rank'])} | {row['content_id']} ──")
    print(f"   Action: {row['action']}")
    print(f"   Why it ranked highly: {impr:,} impressions in 90d, "
          f"not updated in {days} days, position {pos} (striking distance), "
          f"CTR only {ctr_val}%")
    print(f"   Actual status: {actual}")
    print(f"   What would make this wrong: If the page was recently "
          f"restructured without a date update, or if the high impression "
          f"count is from branded queries that won't respond to a refresh, "
          f"or if this client's GA4 data shows strong engagement despite "
          f"the low CTR.")
    print()

── Rank 1 | content_5fe46e04994d ──
   Action: REFRESH
   Why it ranked highly: 517,715 impressions in 90d, not updated in 104 days, position 4.2 (striking distance), CTR only 0.14%
   Actual status: DECLINING
   What would make this wrong: If the page was recently restructured without a date update, or if the high impression count is from branded queries that won't respond to a refresh, or if this client's GA4 data shows strong engagement despite the low CTR.

── Rank 2 | content_2c2606c5d176 ──
   Action: REFRESH
   Why it ranked highly: 347,399 impressions in 90d, not updated in 104 days, position 4.2 (striking distance), CTR only 0.53%
   Actual status: DECLINING
   What would make this wrong: If the page was recently restructured without a date update, or if the high impression count is from branded queries that won't respond to a refresh, or if this client's GA4 data shows strong engagement despite the low CTR.

── Rank 3 | content_cb112fce36be ──
   Action: REFRESH
   Why it ran

---
## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# ── 4a. Identify weak picks in the top 10 ──────────────────────
weak = top10[top10["is_declining"] == 0]

if len(weak) > 0:
    print(f"Found {len(weak)} weak pick(s) in the top 10 — pages the rule")
    print(f"flagged as REFRESH candidates but that are NOT actually declining:\n")
    for _, row in weak.iterrows():
        print(f"  Rank {int(row['rank'])}: {row['content_id']}")
        print(f"    days_since_last_update = {int(row['days_since_last_update'])}, "
              f"impressions_90d = {int(row['impressions_90d']):,}, "
              f"avg_position = {row['avg_position']}")
        print(f"    Why it's questionable: This page meets every staleness and ")
        print(f"    visibility criterion, yet it is NOT declining. The high")
        print(f"    impression count pushed it to the top of the queue, but")
        print(f"    the page may be holding steady because the content is")
        print(f"    evergreen or the keyword is non-competitive. A refresh")
        print(f"    would be wasted effort on a page that isn't losing traffic.")
        print()
else:
    print("No weak picks found in top 10 — this is suspicious.")
    print("Expanding search to top 20...")
    top20 = df_ranked.head(20)
    weak20 = top20[top20["is_declining"] == 0]
    for _, row in weak20.iterrows():
        print(f"  Rank {int(row['rank'])}: {row['content_id']}")
        print(f"    days_since_last_update = {int(row['days_since_last_update'])}, "
              f"impressions_90d = {int(row['impressions_90d']):,}, "
              f"avg_position = {row['avg_position']}")
        print(f"    Why it's questionable: Meets all criteria but NOT declining.")
        print(f"    High impressions inflated the score, but the page is stable.")
        print()

Found 5 weak pick(s) in the top 10 — pages the rule
flagged as REFRESH candidates but that are NOT actually declining:

  Rank 4: content_36ff89c8214e
    days_since_last_update = 104, impressions_90d = 295,097, avg_position = 7.3
    Why it's questionable: This page meets every staleness and 
    visibility criterion, yet it is NOT declining. The high
    impression count pushed it to the top of the queue, but
    the page may be holding steady because the content is
    evergreen or the keyword is non-competitive. A refresh
    would be wasted effort on a page that isn't losing traffic.

  Rank 5: content_c21024970297
    days_since_last_update = 104, impressions_90d = 211,366, avg_position = 5.1
    Why it's questionable: This page meets every staleness and 
    visibility criterion, yet it is NOT declining. The high
    impression count pushed it to the top of the queue, but
    the page may be holding steady because the content is
    evergreen or the keyword is non-competitive. A

In [12]:
# ── 4b. Leakage check ──────────────────────────────────────────
display(Markdown("""
### Leakage check — PASSED ✓

The score uses **only** these inputs:
- `days_since_last_update` — a historical content property (when it was last edited)
- `impressions_90d` — trailing 90-day search impressions (observation window)
- `avg_position` — trailing 90-day average search position (observation window)

**Not used (correctly excluded):**
- `trend_direction` — label source, never a feature ✓
- `trend_pct` — label source, never a feature ✓
- `is_declining` / `is_declining_label` — the target itself ✓
- `content_id` / `client_id` — context only, not features ✓
- `provider_used` / `model_used` — not model features per data dictionary ✓

No future-window or label-derived information leaked into the score.
"""))


### Leakage check — PASSED ✓

The score uses **only** these inputs:
- `days_since_last_update` — a historical content property (when it was last edited)
- `impressions_90d` — trailing 90-day search impressions (observation window)
- `avg_position` — trailing 90-day average search position (observation window)

**Not used (correctly excluded):**
- `trend_direction` — label source, never a feature ✓
- `trend_pct` — label source, never a feature ✓
- `is_declining` / `is_declining_label` — the target itself ✓
- `content_id` / `client_id` — context only, not features ✓
- `provider_used` / `model_used` — not model features per data dictionary ✓

No future-window or label-derived information leaked into the score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.